In [0]:
# ============================================================
# NOTEBOOK: nb_01_Ingestion
# PURPOSE:  Reads the ingestion configuration table, validates
#           every required source file in the raw landing
#           volume, computes content fingerprints, and records
#           the file manifest.
#
#           This notebook does NOT transform data. It confirms
#           the raw layer is complete, readable, and unchanged
#           since the last successful run.
#
# CATALOG:  ktu_assessment_dev
# COMPUTE:  Serverless
# INPUT:    audit.ingestion_config, volume raw_landing
# OUTPUT:   audit.file_manifest, audit.pipeline_run_log,
#           audit.data_quality_results
# ============================================================

# ------------------------------------------------------------
# IMPORTS
# hashlib  = SHA-256 fingerprint of file content
# uuid     = run_id generation
# datetime = timestamps for audit
# os.path  = basename extraction from volume paths
# ------------------------------------------------------------
import hashlib
import uuid
from datetime import datetime
from os.path import basename

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------
CATALOG      = "ktu_assessment_dev"
AUDIT_SCHEMA = "audit"

VOLUME_PATH  = "/Volumes/ktu_assessment_dev/bronze/raw_landing"
SOURCE_ROOT  = f"{VOLUME_PATH}/Training Data"

# Reporting period enforced downstream. Declared here so the
# ingestion layer and every downstream layer reference the
# same window.
REPORTING_PERIOD_START = "2025-01-01"
REPORTING_PERIOD_END   = "2025-12-31"

DEBUG = 1

# ------------------------------------------------------------
# RUN HEADER
# ------------------------------------------------------------
run_id     = str(uuid.uuid4())
notebook   = "nb_01_Ingestion"
start_time = datetime.now()

if DEBUG:
    print("=" * 50)
    print("NB_01_INGESTION STARTED")
    print("=" * 50)
    print(f"Run ID           : {run_id}")
    print(f"Source Root      : {SOURCE_ROOT}")
    print(f"Reporting Period : {REPORTING_PERIOD_START} to {REPORTING_PERIOD_END}")
    print(f"Start Time       : {start_time}")

NB_01_INGESTION STARTED
Run ID           : ac9b1ebe-25c6-4092-b393-a4a43cff4677
Source Root      : /Volumes/ktu_assessment_dev/bronze/raw_landing/Training Data
Reporting Period : 2025-01-01 to 2025-12-31
Start Time       : 2026-09-12 13:17:08.582742


In [0]:
# ============================================================
# AUDIT START AND CONFIG LOAD
# ============================================================
#
# WHAT THIS CELL DOES:
# Writes a RUNNING row to pipeline_run_log before any work
# begins. If the notebook fails mid-execution, the audit
# trail shows an incomplete run rather than no record at all.
#
# Then loads the ingestion configuration. Every source the
# pipeline will process is defined there. No source list is
# hard-coded in this notebook.
# ============================================================

spark.sql(f"""
    INSERT INTO {CATALOG}.{AUDIT_SCHEMA}.pipeline_run_log
    VALUES (
        '{run_id}',
        '{notebook}',
        'bronze',
        'raw_landing',
        '{start_time.strftime("%Y-%m-%d %H:%M:%S")}',
        NULL,
        'RUNNING',
        0,
        0,
        0,
        'Ingestion in progress',
        0
    )
""")

config_df = spark.sql(f"""
    SELECT
        source_name,
        source_file,
        source_subfolder,
        source_format,
        sheet_name,
        header_row,
        target_table,
        processing_order,
        notes
    FROM {CATALOG}.{AUDIT_SCHEMA}.ingestion_config
    WHERE active_flag = true
    ORDER BY processing_order
""")

config_rows = config_df.collect()

if DEBUG:
    print("=" * 50)
    print("INGESTION CONFIG LOADED")
    print("=" * 50)
    print(f"Active sources : {len(config_rows)}")
    print()
    config_df.select(
        "source_name", "sheet_name", "source_format", "processing_order"
    ).show(truncate=False)

INGESTION CONFIG LOADED
Active sources : 9

+---------------+-------------------+-------------+----------------+
|source_name    |sheet_name         |source_format|processing_order|
+---------------+-------------------+-------------+----------------+
|capturing_tool |Capturer1          |xlsx         |10              |
|capturing_tool |Capturer2          |xlsx         |11              |
|capturing_tool |Capturer3          |xlsx         |12              |
|chw_west_coast |Training Attendance|xlsx         |20              |
|chw_kess       |Training Attendance|xlsx         |21              |
|chw_witzenberg |Training Attendance|xlsx         |22              |
|online_export  |(csv)              |csv          |30              |
|lookup_courses |LU_Courses         |xlsx         |40              |
|lookup_facility|LU_Facility        |xlsx         |41              |
+---------------+-------------------+-------------+----------------+



In [0]:
# ============================================================
# SOURCE FILE VALIDATION
# ============================================================
#
# WHAT THIS CELL DOES:
# For every active source in the config, resolves the full
# volume path, confirms the file exists, confirms it is not
# empty, and confirms it is readable by attempting to list
# and stat it.
#
# WHY THIS FAILS LOUDLY:
# The assessment requires the pipeline to fail if a required
# file is missing, empty, unreadable, or structurally
# unexpected. Silent continuation would produce incomplete
# reports without warning.
#
# WHY WE VALIDATE EVEN IF THE FILE WAS SEEN LAST RUN:
# A previous successful run does not guarantee the file is
# still present. Source volumes can be emptied between runs.
# The check is cheap and the failure mode is severe.
#
# RESULT:
# A list of resolved paths and sizes, one entry per unique
# source file. Multiple config rows can reference the same
# file (the capturing tool sheets share one file), so we
# deduplicate by file path.
# ============================================================

validation_errors  = []
resolved_files     = {}

for row in config_rows:
    subfolder = row["source_subfolder"]
    file_name = row["source_file"]

    if subfolder:
        full_path = f"{SOURCE_ROOT}/{subfolder}/{file_name}"
    else:
        full_path = f"{SOURCE_ROOT}/{file_name}"

    # Deduplicate. Multiple config rows may point at one file.
    if full_path in resolved_files:
        continue

    # --------------------------------------------------------
    # EXISTENCE CHECK
    # --------------------------------------------------------
    try:
        entries = dbutils.fs.ls(full_path)
    except Exception as exc:
        validation_errors.append(
            f"MISSING OR UNREADABLE: {full_path} ({exc})"
        )
        continue

    # --------------------------------------------------------
    # EMPTY CHECK
    # dbutils.fs.ls on a file returns one entry with the file
    # itself. Size zero means the file exists but is empty.
    # --------------------------------------------------------
    if not entries:
        validation_errors.append(f"EMPTY: {full_path}")
        continue

    file_entry = entries[0]
    if file_entry.size == 0:
        validation_errors.append(f"EMPTY: {full_path}")
        continue

    resolved_files[full_path] = {
        "path": full_path,
        "name": basename(full_path),
        "size": file_entry.size,
    }

if DEBUG:
    print("=" * 50)
    print("SOURCE FILE VALIDATION")
    print("=" * 50)
    print(f"Config rows           : {len(config_rows)}")
    print(f"Unique files resolved : {len(resolved_files)}")
    print(f"Validation errors     : {len(validation_errors)}")
    print()
    for path, info in sorted(resolved_files.items()):
        print(f"  OK    {info['name']}  ({info['size']} bytes)")
    for err in validation_errors:
        print(f"  ERROR {err}")

# ------------------------------------------------------------
# FAIL LOUDLY IF ANY REQUIRED FILE IS MISSING OR EMPTY
# ------------------------------------------------------------
if validation_errors:
    raise RuntimeError(
        f"Ingestion cannot proceed. {len(validation_errors)} "
        f"source file validation error(s): " + "; ".join(validation_errors)
    )

SOURCE FILE VALIDATION
Config rows           : 9
Unique files resolved : 6
Validation errors     : 0

  OK    Capturing Tool_V1c V2-27 January 2026_SCRUBBED.xlsx  (2152302 bytes)
  OK    CHW Training Attendance-West Coast-Oct 2025_SCRUBBED.xlsx  (110102 bytes)
  OK    CHW Training Attendance_KESS_December 2025_SCRUBBED.xlsx  (129606 bytes)
  OK    WITZENBERG - July-Dec 2025_SCRUBBED.xlsx  (109176 bytes)
  OK    Course and Facility Look Ups.xlsx  (118517 bytes)
  OK    Online Data Export 17 Dec_SCRUBBED.csv  (10845781 bytes)


In [0]:
# ============================================================
# FILE FINGERPRINTS
# ============================================================
#
# WHAT THIS CELL DOES:
# Computes a content-based fingerprint of every resolved
# source file. The fingerprint is the basis of idempotency.
#
# WHY NOT PYTHON open() AND hashlib:
# Python file I/O over a Unity Catalog volume mount is slow.
# The 10.8 MB CSV alone added ~20 minutes on serverless in
# the first run. Spark's binaryFile reader reads the same
# content through the native filesystem layer, which is
# significantly faster on volume-mounted paths.
#
# WHY xxhash64 AND NOT SHA-256:
# SHA-256 requires reading the file in full via Python.
# xxhash64 is a non-cryptographic 64-bit hash that Spark
# computes natively. It is sufficient for detecting content
# change in this assessment context. If cryptographic
# strength were required, Spark's sha2() function could
# replace xxhash64 with the same read strategy.
#
# WHAT WE ACTUALLY READ:
# The binaryFile reader loads the file as a single binary
# column. We apply xxhash64 to that column. The result is
# one row per file with a stable 64-bit hash.
# ============================================================

from pyspark.sql import functions as F

fingerprints = {}

for path, info in resolved_files.items():
    # read binaryFile returns a DataFrame with columns:
    #   path, modificationTime, length, content
    # We hash the content column and take the single value.
    hash_row = (
        spark.read.format("binaryFile")
             .load(path)
             .select(F.xxhash64(F.col("content")).alias("h"))
             .collect()
    )

    if hash_row:
        digest = str(hash_row[0]["h"])
    else:
        digest = "EMPTY"

    fingerprints[path] = digest

    if DEBUG:
        print(f"  {info['name']:<70} {digest}")

if DEBUG:
    print()
    print("=" * 50)
    print("FINGERPRINTS COMPUTED")
    print("=" * 50)
    print(f"Files fingerprinted : {len(fingerprints)}")

  Capturing Tool_V1c V2-27 January 2026_SCRUBBED.xlsx                    -4985641948406252824
  CHW Training Attendance-West Coast-Oct 2025_SCRUBBED.xlsx              -4922987403702069877
  CHW Training Attendance_KESS_December 2025_SCRUBBED.xlsx               2803542552409370738
  WITZENBERG - July-Dec 2025_SCRUBBED.xlsx                               -8405461905426231748
  Online Data Export 17 Dec_SCRUBBED.csv                                 1662990164538964294
  Course and Facility Look Ups.xlsx                                      2731780190603425134

FINGERPRINTS COMPUTED
Files fingerprinted : 6


In [0]:
# ============================================================
# MANIFEST COMPARISON — IDEMPOTENCY LOGIC
# ============================================================
#
# WHAT THIS CELL DOES:
# Loads the current file_manifest, compares each resolved
# file's fingerprint against the manifest, and classifies
# each file as NEW, UNCHANGED, or CHANGED.
#
# CLASSIFICATION:
#   NEW       - file path not seen before, or seen but with
#               a different fingerprint
#   UNCHANGED - file path seen with the same fingerprint
#
# WHY BOTH ARE INGESTED ANYWAY:
# The ingestion layer is the load mechanism, not the
# transform. Bronze is written by nb_02, and Bronze is
# idempotent by overwrite. Therefore this notebook can safely
# re-ingest unchanged files without creating duplicates.
#
# HOW IDEMPOTENCY IS ACTUALLY ACHIEVED:
#   - Bronze tables are overwritten on each run in nb_02
#     rather than appended.
#   - file_manifest records ingest_count and last_ingested_at
#     for observability, not for gating.
#
# WHY WE STILL CLASSIFY:
# A reviewer needs to see, per run, whether each source
# changed since the last run. Silent changes and silent lack
# of changes are both worth knowing.
# ============================================================

existing_manifest = {
    row["file_path"]: row.asDict()
    for row in spark.sql(f"""
        SELECT file_path, fingerprint, ingest_count, first_seen_at, last_ingested_at
        FROM {CATALOG}.{AUDIT_SCHEMA}.file_manifest
    """).collect()
}

classification = []

for path, digest in fingerprints.items():
    prior = existing_manifest.get(path)

    if prior is None:
        status = "NEW"
    elif prior["fingerprint"] == digest:
        status = "UNCHANGED"
    else:
        status = "CHANGED"

    classification.append({
        "file_path":    path,
        "file_name":    resolved_files[path]["name"],
        "file_size":    resolved_files[path]["size"],
        "fingerprint":  digest,
        "status":       status,
        "prior_count":  prior["ingest_count"] if prior else 0,
    })

if DEBUG:
    print("=" * 50)
    print("MANIFEST COMPARISON")
    print("=" * 50)
    for entry in classification:
        print(f"  {entry['status']:<10} {entry['file_name']}")
    print()
    new_count       = sum(1 for c in classification if c["status"] == "NEW")
    changed_count   = sum(1 for c in classification if c["status"] == "CHANGED")
    unchanged_count = sum(1 for c in classification if c["status"] == "UNCHANGED")
    print(f"  NEW       : {new_count}")
    print(f"  CHANGED   : {changed_count}")
    print(f"  UNCHANGED : {unchanged_count}")

MANIFEST COMPARISON
  CHANGED    Capturing Tool_V1c V2-27 January 2026_SCRUBBED.xlsx
  CHANGED    CHW Training Attendance-West Coast-Oct 2025_SCRUBBED.xlsx
  CHANGED    CHW Training Attendance_KESS_December 2025_SCRUBBED.xlsx
  CHANGED    WITZENBERG - July-Dec 2025_SCRUBBED.xlsx
  CHANGED    Online Data Export 17 Dec_SCRUBBED.csv
  CHANGED    Course and Facility Look Ups.xlsx

  NEW       : 0
  CHANGED   : 6
  UNCHANGED : 0


In [0]:
# ============================================================
# MANIFEST UPDATE AND DQ RESULTS
# ============================================================
#
# WHAT THIS CELL DOES:
# Upserts every resolved file into file_manifest via SQL MERGE,
# and writes a data_quality_results row recording the file
# validation outcome.
#
# WHY PYTHON datetime OBJECTS AND NOT STRINGS:
# spark.createDataFrame under Spark Connect (used by
# serverless compute) asserts that any column declared as
# TimestampType receives an actual datetime.datetime object.
# Passing a formatted string raises:
#   AssertionError in conversion.convert_timestamp
# We therefore build the DataFrame with real datetime values,
# and let Spark format them on write.
#
# WHY SQL MERGE AND NOT THE DELTA PYTHON API:
# DeltaTable.forName().merge() asserts against a target that
# has never had a write commit. file_manifest is in that
# state on the first run. SQL MERGE via temp view does not
# hit that code path.
# ============================================================

from pyspark.sql.types import (
    StructType, StructField, StringType, LongType,
    IntegerType, TimestampType
)
from pyspark.sql import Row
from datetime import datetime

manifest_table = f"{CATALOG}.{AUDIT_SCHEMA}.file_manifest"

# ------------------------------------------------------------
# BUILD STAGING ROWS
# Real datetime objects for timestamp columns.
# Real ints for numeric columns.
# Real strings for string columns.
# ------------------------------------------------------------
now_ts = datetime.now()
staged_records = []

for entry in classification:
    prior = existing_manifest.get(entry["file_path"])

    source_name = ""
    for r in config_rows:
        if r["source_file"] == entry["file_name"]:
            source_name = r["source_name"] or ""
            break

    if prior is None:
        first_seen_ts    = now_ts
        ingest_count_val = 1
    else:
        first_seen_ts    = prior["first_seen_at"]
        ingest_count_val = int(prior["ingest_count"]) + 1

    staged_records.append(Row(
        file_name        = str(entry["file_name"]),
        file_path        = str(entry["file_path"]),
        file_size_bytes  = int(entry["file_size"]),
        fingerprint      = str(entry["fingerprint"]),
        first_seen_at    = first_seen_ts,
        last_ingested_at = now_ts,
        ingest_count     = ingest_count_val,
        source_name      = source_name,
    ))

manifest_schema = StructType([
    StructField("file_name",        StringType(),    False),
    StructField("file_path",        StringType(),    False),
    StructField("file_size_bytes",  LongType(),      False),
    StructField("fingerprint",      StringType(),    False),
    StructField("first_seen_at",    TimestampType(), False),
    StructField("last_ingested_at", TimestampType(), False),
    StructField("ingest_count",     IntegerType(),   False),
    StructField("source_name",      StringType(),    True),
])

manifest_df = spark.createDataFrame(staged_records, schema=manifest_schema)

manifest_df.createOrReplaceTempView("staged_file_manifest")

# ------------------------------------------------------------
# SQL MERGE
# ------------------------------------------------------------
spark.sql(f"""
    MERGE INTO {manifest_table} AS t
    USING staged_file_manifest AS s
    ON t.file_path = s.file_path

    WHEN MATCHED THEN UPDATE SET
        t.file_size_bytes  = s.file_size_bytes,
        t.fingerprint      = s.fingerprint,
        t.last_ingested_at = s.last_ingested_at,
        t.ingest_count     = s.ingest_count,
        t.source_name      = s.source_name

    WHEN NOT MATCHED THEN INSERT (
        file_name, file_path, file_size_bytes, fingerprint,
        first_seen_at, last_ingested_at, ingest_count, source_name
    )
    VALUES (
        s.file_name, s.file_path, s.file_size_bytes, s.fingerprint,
        s.first_seen_at, s.last_ingested_at, s.ingest_count, s.source_name
    )
""")

# ------------------------------------------------------------
# DATA QUALITY RESULT ROW
# ------------------------------------------------------------
validated = len(resolved_files)
expected  = len(set(
    f"{r['source_subfolder']}/{r['source_file']}"
    if r["source_subfolder"] else r["source_file"]
    for r in config_rows
))

spark.sql(f"""
    INSERT INTO {CATALOG}.{AUDIT_SCHEMA}.data_quality_results
    VALUES (
        '{run_id}',
        'ingestion_file_validation',
        'completeness',
        'raw_landing',
        'All required source files present and non-empty',
        '{validated} unique files resolved from {expected} configured paths',
        'PASS',
        NULL,
        current_timestamp()
    )
""")

if DEBUG:
    print("=" * 50)
    print("MANIFEST UPDATED")
    print("=" * 50)
    spark.sql(f"""
        SELECT file_name, ingest_count, last_ingested_at
        FROM {manifest_table}
        ORDER BY file_name
    """).show(truncate=False)

MANIFEST UPDATED
+---------------------------------------------------------+------------+-------------------------+
|file_name                                                |ingest_count|last_ingested_at         |
+---------------------------------------------------------+------------+-------------------------+
|CHW Training Attendance-West Coast-Oct 2025_SCRUBBED.xlsx|2           |2026-09-12 13:17:17.90248|
|CHW Training Attendance_KESS_December 2025_SCRUBBED.xlsx |2           |2026-09-12 13:17:17.90248|
|Capturing Tool_V1c V2-27 January 2026_SCRUBBED.xlsx      |2           |2026-09-12 13:17:17.90248|
|Course and Facility Look Ups.xlsx                        |2           |2026-09-12 13:17:17.90248|
|Online Data Export 17 Dec_SCRUBBED.csv                   |2           |2026-09-12 13:17:17.90248|
|WITZENBERG - July-Dec 2025_SCRUBBED.xlsx                 |2           |2026-09-12 13:17:17.90248|
+---------------------------------------------------------+------------+--------------------

In [0]:
# ============================================================
# FINALISE AUDIT
# ============================================================
#
# WHAT THIS CELL DOES:
# Updates the pipeline_run_log row created in Cell 2 with the
# final status, row counts, and duration, then prints a
# summary in the standard banner format.
#
# WHY UPDATE AND NOT INSERT:
# Cell 2 inserted a RUNNING row so the audit trail shows the
# notebook started. This cell closes that same row with the
# final outcome, so the run appears as one complete record
# rather than two disconnected entries.
# ============================================================

end_time = datetime.now()
duration = int((end_time - start_time).total_seconds())

total_files     = len(resolved_files)
total_new       = sum(1 for c in classification if c["status"] == "NEW")
total_changed   = sum(1 for c in classification if c["status"] == "CHANGED")
total_unchanged = sum(1 for c in classification if c["status"] == "UNCHANGED")

message = (
    f"Ingestion validated {total_files} source files. "
    f"NEW: {total_new}, CHANGED: {total_changed}, UNCHANGED: {total_unchanged}."
)

spark.sql(f"""
    UPDATE {CATALOG}.{AUDIT_SCHEMA}.pipeline_run_log
    SET
        end_time         = '{end_time.strftime("%Y-%m-%d %H:%M:%S")}',
        status           = 'SUCCESS',
        rows_in          = {total_files},
        rows_out         = {total_files},
        rows_rejected    = 0,
        message          = '{message}',
        duration_seconds = {duration}
    WHERE run_id = '{run_id}'
""")

if DEBUG:
    print("=" * 50)
    print("INGESTION SUMMARY")
    print("=" * 50)
    print(f"Run ID             : {run_id}")
    print(f"Files validated    : {total_files}")
    print(f"  NEW              : {total_new}")
    print(f"  CHANGED          : {total_changed}")
    print(f"  UNCHANGED        : {total_unchanged}")
    print(f"Duration           : {duration}s")
    print(f"Status             : SUCCESS")
    print("=" * 50)

print("nb_01_Ingestion completed successfully.")

INGESTION SUMMARY
Run ID             : ac9b1ebe-25c6-4092-b393-a4a43cff4677
Files validated    : 6
  NEW              : 0
  CHANGED          : 6
  UNCHANGED        : 0
Duration           : 15s
Status             : SUCCESS
nb_01_Ingestion completed successfully.


In [0]:
# ============================================================
# HOUSEKEEPING: CLOSE ORPHANED RUNNING ROWS
# ============================================================
#
# WHAT THIS CELL DOES:
# The first attempt at nb_01_Ingestion failed in Cell 6 and
# left a RUNNING row in pipeline_run_log that will never be
# closed. This cell marks any such orphaned rows as FAILED,
# so the audit trail is clean: one FAILED attempt, one
# SUCCESS attempt.
#
# WHY THIS MATTERS:
# A reviewer scanning pipeline_run_log should see every row
# resolved to a terminal state. Orphaned RUNNING rows make
# it impossible to tell whether the run is still executing
# or simply died.
#
# WHY IT EXCLUDES THE CURRENT RUN:
# The current run's RUNNING row is also closed by Cell 7.
# This cell explicitly excludes the current run_id so it
# does not interfere with the row that Cell 7 just updated.
# ============================================================

spark.sql(f"""
    UPDATE {CATALOG}.{AUDIT_SCHEMA}.pipeline_run_log
    SET
        status           = 'FAILED',
        message          = 'Orphaned RUNNING row closed by housekeeping. Original run did not reach its finalise step.',
        end_time         = current_timestamp(),
        duration_seconds = 0
    WHERE status        = 'RUNNING'
      AND notebook_name = 'nb_01_Ingestion'
      AND run_id       <> '{run_id}'
""")

if DEBUG:
    print("=" * 50)
    print("HOUSEKEEPING COMPLETE")
    print("=" * 50)
    spark.sql(f"""
        SELECT run_id, status, start_time, end_time, message
        FROM {CATALOG}.{AUDIT_SCHEMA}.pipeline_run_log
        WHERE notebook_name = 'nb_01_Ingestion'
        ORDER BY start_time
    """).show(truncate=False)

HOUSEKEEPING COMPLETE
+------------------------------------+-------+-------------------+-------------------+---------------------------------------------------------------------+
|run_id                              |status |start_time         |end_time           |message                                                              |
+------------------------------------+-------+-------------------+-------------------+---------------------------------------------------------------------+
|bc61192a-e916-4fde-acd1-72edef793ced|SUCCESS|2026-09-12 12:38:38|2026-09-12 13:11:00|Ingestion validated 6 source files. NEW: 6, CHANGED: 0, UNCHANGED: 0.|
|ac9b1ebe-25c6-4092-b393-a4a43cff4677|SUCCESS|2026-09-12 13:17:08|2026-09-12 13:17:23|Ingestion validated 6 source files. NEW: 0, CHANGED: 6, UNCHANGED: 0.|
+------------------------------------+-------+-------------------+-------------------+---------------------------------------------------------------------+



In [0]:
# ============================================================
# INGESTION VERIFICATION
# ============================================================
#
# WHAT THIS CELL DOES:
# Produces a reviewer-facing summary of the ingestion run:
# what files were processed, what the manifest contains,
# what the audit log shows, and what DQ checks were recorded.
#
# This is the evidence a senior reviewer uses to confirm the
# ingestion is trustworthy before trusting the downstream
# layers.
# ============================================================

if DEBUG:
    print("=" * 70)
    print("INGESTION VERIFICATION")
    print("=" * 70)
    print()

    print("FILE MANIFEST")
    print("-" * 70)
    spark.sql(f"""
        SELECT file_name, file_size_bytes, ingest_count, source_name
        FROM {CATALOG}.{AUDIT_SCHEMA}.file_manifest
        ORDER BY file_name
    """).show(truncate=False)

    print("PIPELINE RUN LOG (this notebook)")
    print("-" * 70)
    spark.sql(f"""
        SELECT run_id, status, rows_in, rows_out, rows_rejected, duration_seconds
        FROM {CATALOG}.{AUDIT_SCHEMA}.pipeline_run_log
        WHERE notebook_name = 'nb_01_Ingestion'
        ORDER BY start_time
    """).show(truncate=False)

    print("DATA QUALITY RESULTS (this run)")
    print("-" * 70)
    spark.sql(f"""
        SELECT check_name, check_category, status, actual
        FROM {CATALOG}.{AUDIT_SCHEMA}.data_quality_results
        WHERE run_id = '{run_id}'
        ORDER BY check_ts
    """).show(truncate=False)

    manifest_count = spark.sql(
        f"SELECT COUNT(*) AS n FROM {CATALOG}.{AUDIT_SCHEMA}.file_manifest"
    ).collect()[0]["n"]

    dq_pass_count = spark.sql(
        f"SELECT COUNT(*) AS n FROM {CATALOG}.{AUDIT_SCHEMA}.data_quality_results "
        f"WHERE run_id = '{run_id}' AND status = 'PASS'"
    ).collect()[0]["n"]

    print("=" * 70)
    print(f"Manifest rows      : {manifest_count}")
    print(f"Expected files     : 6")
    print(f"DQ checks passed   : {dq_pass_count}")
    print("=" * 70)

INGESTION VERIFICATION

FILE MANIFEST
----------------------------------------------------------------------
+---------------------------------------------------------+---------------+------------+--------------+
|file_name                                                |file_size_bytes|ingest_count|source_name   |
+---------------------------------------------------------+---------------+------------+--------------+
|CHW Training Attendance-West Coast-Oct 2025_SCRUBBED.xlsx|110102         |2           |chw_west_coast|
|CHW Training Attendance_KESS_December 2025_SCRUBBED.xlsx |129606         |2           |chw_kess      |
|Capturing Tool_V1c V2-27 January 2026_SCRUBBED.xlsx      |2152302        |2           |capturing_tool|
|Course and Facility Look Ups.xlsx                        |118517         |2           |lookup_courses|
|Online Data Export 17 Dec_SCRUBBED.csv                   |10845781       |2           |online_export |
|WITZENBERG - July-Dec 2025_SCRUBBED.xlsx                 |